# Block 1: Basic Elements in Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_1_4_optimizers.ipynb)

## 1.4: Optimizers

This short example brings together the basic elements of a neural network:

* **MLP**: Map two input features to a class prediction.
* **Loss**: Measure how well the predictions match the labels.
* **Backpropagation**: Compute gradients with `loss.backward()`.
* **Optimization**: Update the parameters with `optimizer.step()`.

We will train a small classifier on two moons and visualize its decision regions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons

torch.manual_seed(7)

### 1. The Problem: Two Moons

Each point has two coordinates and belongs to one of two classes. A straight line cannot separate the two moons well: we need a non-linear classifier.

`TensorDataset` pairs each input with its label. `DataLoader` groups these pairs into mini-batches and shuffles their order each epoch.

In [ ]:
X_np, y_np = make_moons(n_samples=640, noise=0.15, random_state=7)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)  # Shape: (640, 1)

dataset = TensorDataset(X, y)
print(f"{len(dataset)=}")
for i in range(10,15):
    print(f"{i=}, {dataset[i]=}")

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=y[:, 0], cmap="RdBu", s=20)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Two-moons binary classification")
plt.show()

### 2. Model, Loss and Optimizer

Our MLP has two hidden layers with ReLU activations and one output **logit** per example. `BCEWithLogitsLoss` combines sigmoid and binary cross-entropy, so we do not add a sigmoid layer to the model.

We use SGD with momentum and a fixed learning rate. The optimizer receives the model's parameters: these are the tensors it will update.

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

### 3. Train with Mini-Batches

An **epoch** is one pass through the training data. Each mini-batch produces a loss, a set of gradients and one parameter update. We clear the previous gradients before each forward pass because PyTorch accumulates gradients by default.

In [ ]:
epochs = 100
loss_history = []

model.train()
for epoch in range(epochs):
    total_loss = 0.0

    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        optimizer.zero_grad()                 # Clear previous gradients
        logits = model(x_batch)               # Forward pass
        loss = criterion(logits, y_batch)     # Compute the loss
        loss.backward()                       # Compute gradients
        optimizer.step()                      # Update the parameters

        if epoch == 0 and batch_idx == 0:
            print(f"Input batch: {x_batch.shape}")
            print(f"Output logits: {logits.shape}")
            print(f"Target batch: {y_batch.shape}")
            print()

        total_loss += loss.item() * len(x_batch)

    loss_history.append(total_loss / len(X))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:2d}: mean training loss = {loss_history[-1]:.4f}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(range(1, epochs + 1), loss_history)
plt.xlabel("Epoch")
plt.ylabel("Mean mini-batch loss")
plt.title("Training progress")
plt.grid(alpha=0.25)
plt.show()

**Question:** Which line actually changes the weights?

**Question:** How many times are parameters updated during this training?

### 4. Visualize the Learned Decision Regions

We evaluate the model on a grid of input coordinates. Applying sigmoid to its logits gives the predicted probability of class 1; the decision boundary is where this probability equals 0.5.

`model.eval()` selects evaluation mode; `torch.no_grad()` disables gradient tracking for this prediction step. The points shown are the **training data**: this plot illustrates what the model fitted, rather than measuring performance on unseen examples.

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X_np[:, 0].min() - 0.4, X_np[:, 0].max() + 0.4, 200),
    np.linspace(X_np[:, 1].min() - 0.4, X_np[:, 1].max() + 0.4, 200),
)
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

plt.figure(figsize=(6, 4))
plt.contourf(xx, yy, probabilities, levels=np.linspace(0, 1, 11),
             cmap="RdBu", alpha=0.6)
plt.colorbar(label="Predicted probability of class 1")
plt.contour(xx, yy, probabilities, levels=[0.5], colors="black", linewidths=1.5)
plt.scatter(X[:, 0], X[:, 1], c=y[:, 0], cmap="RdBu", s=20,
            edgecolors="white", linewidths=0.4)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Learned decision regions")
plt.tight_layout()
plt.show()